# Advanced HEN Methods

**Learning outcome:** Apply advanced hen methods through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `solver`  
**Expected runtime:** 10 to 90 minutes  
**Optional extras:** `synthesis`

## Study question and data

**Study question:** How do alternative synthesis and network-improvement methods change the candidate design?

## Step 1: Run enhanced synthesis

This cell loads the four-stream Yee and Grossmann case and runs `enhanced_heat_exchanger_network(quality_tier=1)`. This is enhanced OpenHENS synthesis at an explicit quality tier. `enhanced` holds the result, and `enhanced_ranking` is its best successful outcome.

In [ ]:
from OpenPinch import PinchProblem

problem = PinchProblem(
    "Four-stream-Yee-and-Grossmann-1990-1.json", project_name="Four Stream"
)
enhanced = problem.design.enhanced_heat_exchanger_network(
    quality_tier=1
)
enhanced_ranking = enhanced.top(1)

## Step 2: Compare advanced design methods

This cell runs four further design methods. `open_hens()` runs the tier-one OpenHENS workflow and `pinch_design()` runs the pinch design method alone. `thermal_derivative()` is seeded with the pinch design network, and `network_evolution()` with the thermal-derivative network. Compare the four `*_ranking` outputs.

In [ ]:
open_hens = problem.design.open_hens()
open_hens_ranking = open_hens.top(1)
pinch_design = problem.design.pinch_design()
pinch_ranking = pinch_design.top(1)
thermal = problem.design.thermal_derivative(
    (pinch_design.selected_network,)
)
thermal_ranking = thermal.top(1)
evolution = problem.design.network_evolution(
    (thermal.selected_network,)
)
evolution_ranking = evolution.top(1)

## Step 3: Optimise duties on a hand-built network

This cell defines a network structure by hand: four recovery matches in two stages as `(hot stream, cold stream, stage)`, a heater on Milk Concentrate and a cooler on each hot stream. `optimise_duties()` keeps that structure and allocates duty (and stream splits) three ways. `min_utility` minimises utility use with at least 10 K approach on every exchanger. `min_area` minimises total area with hot utility capped at 1.5 times that minimum. `min_cost` minimises total annual cost, requiring only a 1 K approach. Each result is a normal design view, so `top()`, `grid()` and the utility totals work as for the synthesis methods.

In [ ]:
structure = {
    "recovery": [
        ("Raw Milk", "Milk Concentrate", 1),
        ("HT Flash", "Milk Concentrate", 1),
        ("Raw Milk", "CIP Water", 2),
        ("HT Flash", "CIP Water", 2),
    ],
    "heaters": ["Milk Concentrate"],
    "coolers": ["Raw Milk", "HT Flash"],
}
min_utility = problem.design.optimise_duties(
    structure, objective="utility", min_approach_temperature=10.0
)
min_area = problem.design.optimise_duties(
    structure,
    objective="area",
    min_approach_temperature=10.0,
    max_hot_utility=1.5 * min_utility.total_hot_utility,
)
min_cost = problem.design.optimise_duties(structure, objective="cost")
duty_rankings = [view.top(1) for view in (min_utility, min_area, min_cost)]

## Review the result

Compare the leading result from every method on the same stream and economic basis before attributing design differences to the search method.

In [ ]:
from IPython.display import display

display(enhanced_ranking)
display(open_hens_ranking)
display(pinch_ranking)
display(thermal_ranking)
display(evolution_ranking)
display(duty_rankings)

## Interpret the result

Compare every method on the same streams, utility assumptions, and economic basis; record solver status with each result.

## Adapt this template

Start with one method and small search bounds, then pass only reviewed networks into derivative or evolution workflows.